# LogiFlow-RL: Colab TRL-GRPO Training Notebook

This notebook is a Phase-2 training notebook for the `crisis_logistics_env` environment.

It covers:
- Installing dependencies in Colab
- Loading the environment
- Building a prompt dataset from environment observations
- Running a minimal TRL `GRPOTrainer` loop
- Evaluating before/after behavior in the environment

Recommended GPU: T4 or better.

In [ ]:
%cd crisis_logistics_env
!pip install -e .

In [ ]:
# Colab setup
!pip -q install openenv trl transformers accelerate datasets peft bitsandbytes

In [ ]:
# If running in a fresh Colab session, clone your repo.
# !git clone <YOUR_REPO_URL>
# %cd crisis-logistics-env

import os
import json
import random
import re
from dataclasses import dataclass
from typing import Dict, List

import torch
from datasets import Dataset
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer

from crisis_logistics_env.server.crisis_logistics_env_environment import (
    CrisisLogisticsEnvironment,
    choose_network_action,
)
from crisis_logistics_env.models import CrisisLogisticsAction
from crisis_logistics_env.tasks import list_tasks

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
OUTPUT_DIR = "./outputs/logiflow-grpo"

print("Using model:", MODEL_NAME)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
def build_prompt(observation, task_title: str) -> str:
    return (
        f"Task: {task_title}\n"
        f"Objective: {observation.objective}\n"
        f"Step: {observation.step_count + 1}/{observation.max_steps}\n"
        f"Visible nodes: {observation.visible_node_ids}\n"
        f"Observed node loads: {observation.observed_node_loads}\n"
        f"Node capacities: {observation.node_capacities}\n"
        f"Visible connectivity: {observation.visible_connectivity}\n"
        f"Active disruptions: {observation.active_disruptions}\n"
        f"In-transit shipments: {observation.in_transit_shipments[:8]}\n"
        f"Incoming shipment: source={observation.pending_source_node}, volume={observation.incoming_load}\n"
        f"Traffic event: {observation.event_label}\n"
        "Return exactly one JSON object with keys: reasoning, source_node, dest_node, shipment_volume."
    )


def build_training_rows(samples_per_task: int = 48) -> List[Dict]:
    rows: List[Dict] = []
    for task in list_tasks():
        env = CrisisLogisticsEnvironment()
        obs = env.reset(task_id=task.task_id)
        for _ in range(samples_per_task):
            source = int(obs.pending_source_node)
            valid_dests = obs.connectivity.get(str(source), [])
            rows.append(
                {
                    "prompt": build_prompt(obs, task.title),
                    "source_node": source,
                    "valid_dests": json.dumps(valid_dests),
                    "incoming_load": float(obs.incoming_load),
                }
            )
            obs = env.step(choose_network_action(obs))
            if obs.done:
                break
    return rows


train_rows = build_training_rows(samples_per_task=40)
train_ds = Dataset.from_list(train_rows)
print("Training rows:", len(train_ds))
train_ds[0]

In [ ]:
def extract_json(text: str) -> Dict:
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return {}
    try:
        return json.loads(match.group(0))
    except Exception:
        return {}


def action_reward(completions, source_node=None, valid_dests=None, incoming_load=None, **kwargs):
    rewards = []
    for i, completion in enumerate(completions):
        reward = 0.0
        parsed = extract_json(completion)
        if parsed:
            reward += 0.25

        required = {"reasoning", "source_node", "dest_node", "shipment_volume"}
        if required.issubset(parsed.keys()):
            reward += 0.25

        exp_source = int(source_node[i]) if source_node is not None else None
        try:
            src = int(parsed.get("source_node", -999))
            if exp_source is not None and src == exp_source:
                reward += 0.15
        except Exception:
            pass

        try:
            dest = int(parsed.get("dest_node", -999))
            allowed = json.loads(valid_dests[i]) if valid_dests is not None else []
            if dest in allowed:
                reward += 0.2
        except Exception:
            pass

        try:
            vol = float(parsed.get("shipment_volume", -1))
            if 0 < vol <= 60:
                reward += 0.15
        except Exception:
            pass

        rewards.append(float(min(1.0, max(0.0, reward))))
    return rewards


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_generations=4,
    max_prompt_length=1024,
    max_completion_length=128,
    max_steps=120,
    logging_steps=5,
    save_steps=60,
    report_to=[],
)

trainer = GRPOTrainer(
    model=MODEL_NAME,
    args=training_args,
    train_dataset=train_ds,
    reward_funcs=[action_reward],
    peft_config=peft_config,
)

trainer.train()

In [ ]:
import matplotlib.pyplot as plt
history = [x['rewards/mean'] for x in trainer.state.log_history if 'rewards/mean' in x]
plt.figure(figsize=(10,4))
plt.plot(history)
plt.xlabel('Training Step'); plt.ylabel('Mean Reward')
plt.title('GRPO Training — LogiFlow-RL')
plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight')

In [ ]:
# Save adapter/checkpoint
trainer.save_model(OUTPUT_DIR)
print("Saved checkpoint to", OUTPUT_DIR)

In [ ]:
def generate_action(model, tokenizer, prompt: str) -> CrisisLogisticsAction:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    payload = extract_json(text)
    if not payload:
        return CrisisLogisticsAction(target_hub=0)
    try:
        return CrisisLogisticsAction(**payload)
    except Exception:
        return CrisisLogisticsAction(target_hub=0)


def evaluate_policy(model, tokenizer, task_id: str, max_steps: int = 120):
    env = CrisisLogisticsEnvironment()
    obs = env.reset(task_id=task_id)
    while not obs.done and obs.step_count < max_steps:
        prompt = build_prompt(obs, env.task.title)
        action = generate_action(model, tokenizer, prompt)
        obs = env.step(action)
    return {
        "task_id": task_id,
        "score": env.score,
        "total_reward": env.total_reward,
        "retail_delivered": env.retail_delivered,
        "sla_success_rate": env._sla_success_rate(),
        "bottlenecks": env.bottlenecks,
        "invalid_actions": env.invalid_actions,
    }


In [ ]:
trained_model = trainer.model
for task in list_tasks():
    result = evaluate_policy(trained_model, tokenizer, task.task_id)
    print(result)